# 05 - News Sentiment Data Extraction

## Objective
Build a controlled and reproducible financial news sentiment extraction pipeline using the Alpha Vantage NEWS_SENTIMENT endpoint.

## Key Design Principles
- **Incremental extraction** with progress tracking to handle rate limits
- **Resumable** - can be stopped and restarted without losing progress
- **Local caching** - saves all raw API responses as JSON
- **Rate limit aware** - respects Alpha Vantage free-tier limits

## Progress Tracking
Progress is tracked in `Market_Data/raw/news_sentiment/api_progress.csv`:
- Ticker | Last_Page_Downloaded | Last_Run_Date
- Each successful API call updates the progress file
- Resuming starts from Last_Page_Downloaded + 1

## Data Source
Alpha Vantage NEWS_SENTIMENT API

## Date Range
2023-01-01 to 2026-01-01

## Output
- Progress file: `Market_Data/raw/news_sentiment/api_progress.csv`
- Raw JSON cache: `Market_Data/raw/news_sentiment/raw_json/`
- Aggregated CSV: `Market_Data/raw/news_sentiment.csv`

---
## 1. Environment Setup

In [1]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Load environment variables from .env file
load_dotenv()

# Validate API key
API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")

if not API_KEY:
    raise ValueError(
        "ALPHA_VANTAGE_API_KEY not found in environment variables.\n"
        "Please create a .env file in the ml_pipeline directory with:\n"
        "ALPHA_VANTAGE_API_KEY=your_api_key_here\n\n"
        "Get a free API key at: https://www.alphavantage.co/support/#api-key"
    )

print("API Key loaded and validated successfully")
print(f"Key prefix: {API_KEY[:4]}...{API_KEY[-4:]}")

API Key loaded and validated successfully
Key prefix: X1F6...3PWF


---
## 2. Configuration

In [3]:
# =============================================================================
# DATE RANGE CONFIGURATION
# =============================================================================
START_DATE = "2023-01-01"
END_DATE = "2026-01-01"

# =============================================================================
# TICKER CONFIGURATION
# High-liquidity subset of Nifty 100 for sentiment extraction
# =============================================================================
TICKERS = [
    "NSEI",           # Nifty Index
    "RELIANCE",
    "TCS",
    "INFY",
    "HDFCBANK",
    "ICICIBANK",
    "SBIN",
    "LT",
    "ITC",
    "KOTAKBANK",
    "AXISBANK",
    "BAJFINANCE",
    "HINDUNILVR",
    "ASIANPAINT",
    "MARUTI",
    "SUNPHARMA"
]

# =============================================================================
# API CONFIGURATION
# =============================================================================
BASE_URL = "https://www.alphavantage.co/query"

# Rate limit settings (free tier: 5 calls/minute, 500 calls/day)
RATE_LIMIT_DELAY = 15  # seconds between API calls
MAX_ARTICLES_PER_REQUEST = 200  # Articles per page
MAX_PAGES_PER_TICKER = 10  # Maximum pages to fetch per ticker

# =============================================================================
# PATH CONFIGURATION
# =============================================================================
BASE_DIR = Path("../Market_Data/raw/news_sentiment")
CACHE_DIR = BASE_DIR / "raw_json"
PROGRESS_FILE = BASE_DIR / "api_progress.csv"
OUTPUT_FILE = Path("../Market_Data/raw/news_sentiment.csv")

# Create directories
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded:")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Tickers: {len(TICKERS)} stocks")
print(f"  Max pages per ticker: {MAX_PAGES_PER_TICKER}")
print(f"  Progress file: {PROGRESS_FILE}")
print(f"  Cache directory: {CACHE_DIR}")

Configuration loaded:
  Date range: 2023-01-01 to 2026-01-01
  Tickers: 16 stocks
  Max pages per ticker: 10
  Progress file: ..\Market_Data\raw\news_sentiment\api_progress.csv
  Cache directory: ..\Market_Data\raw\news_sentiment\raw_json


---
## 3. Progress Tracking Functions

In [4]:
def load_progress() -> pd.DataFrame:
    """
    Load progress tracking file.
    Creates a new one if it doesn't exist.
    
    Returns:
        DataFrame with columns: Ticker, Last_Page_Downloaded, Last_Run_Date
    """
    if PROGRESS_FILE.exists():
        progress = pd.read_csv(PROGRESS_FILE)
        print(f"Loaded existing progress file with {len(progress)} tickers")
    else:
        # Initialize progress for all tickers
        progress = pd.DataFrame({
            "Ticker": TICKERS,
            "Last_Page_Downloaded": 0,
            "Last_Run_Date": None
        })
        progress.to_csv(PROGRESS_FILE, index=False)
        print(f"Created new progress file for {len(TICKERS)} tickers")
    
    return progress


def save_progress(progress: pd.DataFrame) -> None:
    """
    Save progress tracking file to disk.
    """
    progress.to_csv(PROGRESS_FILE, index=False)


def update_ticker_progress(progress: pd.DataFrame, ticker: str, page: int) -> pd.DataFrame:
    """
    Update progress for a specific ticker after successful API call.
    
    Args:
        progress: Progress DataFrame
        ticker: Ticker symbol
        page: Page number that was just downloaded
    
    Returns:
        Updated progress DataFrame
    """
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    if ticker in progress["Ticker"].values:
        progress.loc[progress["Ticker"] == ticker, "Last_Page_Downloaded"] = page
        progress.loc[progress["Ticker"] == ticker, "Last_Run_Date"] = now
    else:
        # Add new ticker if not present
        new_row = pd.DataFrame({
            "Ticker": [ticker],
            "Last_Page_Downloaded": [page],
            "Last_Run_Date": [now]
        })
        progress = pd.concat([progress, new_row], ignore_index=True)
    
    # Save immediately after each update
    save_progress(progress)
    
    return progress


def get_ticker_start_page(progress: pd.DataFrame, ticker: str) -> int:
    """
    Get the next page to download for a ticker.
    
    Args:
        progress: Progress DataFrame
        ticker: Ticker symbol
    
    Returns:
        Page number to start from (Last_Page_Downloaded + 1)
    """
    if ticker in progress["Ticker"].values:
        last_page = progress.loc[progress["Ticker"] == ticker, "Last_Page_Downloaded"].iloc[0]
        return int(last_page) + 1
    else:
        return 1


print("Progress tracking functions defined")

Progress tracking functions defined


In [5]:
# Load or create progress file
progress_df = load_progress()
print("\nCurrent progress:")
print(progress_df.to_string())

Loaded existing progress file with 16 tickers

Current progress:
        Ticker  Last_Page_Downloaded        Last_Run_Date
0         NSEI                     2  2026-04-03 14:32:58
1     RELIANCE                     2  2026-04-03 14:33:14
2          TCS                     2  2026-04-03 14:33:29
3         INFY                     2  2026-04-03 14:33:45
4     HDFCBANK                     2  2026-04-03 14:34:01
5    ICICIBANK                     2  2026-04-03 14:34:16
6         SBIN                     2  2026-04-03 14:34:32
7           LT                     2  2026-04-03 14:34:47
8          ITC                     2  2026-04-03 14:35:03
9    KOTAKBANK                     1  2026-04-03 14:27:25
10    AXISBANK                     1  2026-04-03 14:27:40
11  BAJFINANCE                     1  2026-04-03 14:27:56
12  HINDUNILVR                     1  2026-04-03 14:28:11
13  ASIANPAINT                     1  2026-04-03 14:28:27
14      MARUTI                     1  2026-04-03 14:28:42
15   SU

---
## 4. Cache Functions

In [6]:
def get_cache_filename(ticker: str, page: int) -> Path:
    """
    Generate a standardized cache filename for a ticker/page.
    
    Args:
        ticker: Stock ticker symbol
        page: Page number
    
    Returns:
        Path object for the cache file
    """
    filename = f"NEWS_SENTIMENT_{ticker}_page{page:03d}.json"
    return CACHE_DIR / filename


def is_page_cached(ticker: str, page: int) -> bool:
    """
    Check if a cached response exists for the given ticker/page.
    """
    return get_cache_filename(ticker, page).exists()


def load_from_cache(ticker: str, page: int) -> dict:
    """
    Load a cached API response from disk.
    """
    cache_file = get_cache_filename(ticker, page)
    with open(cache_file, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_to_cache(ticker: str, page: int, data: dict) -> None:
    """
    Save an API response to the cache.
    """
    cache_file = get_cache_filename(ticker, page)
    with open(cache_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)


print("Cache functions defined")

Cache functions defined


---
## 5. API Request Functions

In [7]:
def fetch_news_sentiment_page(
    api_key: str,
    ticker: str,
    time_from: str = None,
    time_to: str = None,
    limit: int = 200
) -> dict:
    """
    Fetch a page of news sentiment data from Alpha Vantage API.
    
    Args:
        api_key: Alpha Vantage API key
        ticker: Stock ticker symbol
        time_from: Start time (YYYYMMDDTHHMM format)
        time_to: End time (YYYYMMDDTHHMM format)
        limit: Maximum articles to fetch
    
    Returns:
        API response as dictionary, or error dict on failure
    """
    params = {
        "function": "NEWS_SENTIMENT",
        "tickers": ticker,
        "apikey": api_key,
        "limit": limit,
        "sort": "LATEST"
    }
    
    if time_from:
        params["time_from"] = time_from
    if time_to:
        params["time_to"] = time_to
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Check for API error messages
        if "Note" in data:
            print(f"    Rate limit reached!")
            return {"error": "rate_limit", "message": data["Note"]}
        
        if "Error Message" in data:
            print(f"    API error: {data['Error Message']}")
            return {"error": "api_error", "message": data["Error Message"]}
        
        if "Information" in data:
            print(f"    API info: {data['Information'][:80]}...")
            return {"error": "info", "message": data["Information"]}
        
        return data
        
    except requests.exceptions.Timeout:
        print(f"    Request timeout")
        return {"error": "timeout"}
    except requests.exceptions.HTTPError as e:
        print(f"    HTTP error: {e}")
        return {"error": "http_error", "message": str(e)}
    except requests.exceptions.RequestException as e:
        print(f"    Request failed: {e}")
        return {"error": "request_error", "message": str(e)}
    except json.JSONDecodeError:
        print(f"    Invalid JSON response")
        return {"error": "json_error"}


print("API request function defined")

API request function defined


---
## 6. Incremental Extraction Pipeline

In [8]:
def extract_ticker_sentiment_incremental(
    ticker: str,
    progress: pd.DataFrame,
    api_key: str,
    start_date: str,
    end_date: str,
    max_pages: int = 10,
    rate_limit_delay: int = 15
) -> tuple:
    """
    Extract news sentiment for a ticker incrementally.
    Resumes from last downloaded page.
    
    Args:
        ticker: Stock ticker symbol
        progress: Progress tracking DataFrame
        api_key: Alpha Vantage API key
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        max_pages: Maximum pages to fetch
        rate_limit_delay: Seconds between API calls
    
    Returns:
        Tuple of (updated_progress, pages_downloaded, hit_rate_limit)
    """
    start_page = get_ticker_start_page(progress, ticker)
    
    if start_page > max_pages:
        print(f"  [{ticker}] Already completed ({start_page-1} pages)")
        return progress, 0, False
    
    print(f"  [{ticker}] Starting from page {start_page}")
    
    # Convert dates to Alpha Vantage format
    time_from = datetime.strptime(start_date, "%Y-%m-%d").strftime("%Y%m%dT0000")
    time_to = datetime.strptime(end_date, "%Y-%m-%d").strftime("%Y%m%dT2359")
    
    pages_downloaded = 0
    
    for page in range(start_page, max_pages + 1):
        print(f"    Fetching page {page}...")
        
        # Make API request
        data = fetch_news_sentiment_page(
            api_key, ticker, time_from, time_to, MAX_ARTICLES_PER_REQUEST
        )
        
        # Check for rate limit
        if data and data.get("error") == "rate_limit":
            print(f"    Rate limit hit! Stopping extraction.")
            return progress, pages_downloaded, True
        
        # Check for other errors
        if data and "error" in data:
            print(f"    Error on page {page}, skipping ticker")
            break
        
        # Cache successful response
        if data and "feed" in data:
            article_count = len(data.get("feed", []))
            save_to_cache(ticker, page, data)
            print(f"    Cached {article_count} articles")
            
            # Update progress immediately
            progress = update_ticker_progress(progress, ticker, page)
            pages_downloaded += 1
            
            # If fewer articles than limit, no more pages
            if article_count < MAX_ARTICLES_PER_REQUEST:
                print(f"    No more articles available")
                break
        else:
            print(f"    No feed in response")
            break
        
        # Rate limiting between pages
        if page < max_pages:
            print(f"    Waiting {rate_limit_delay}s...")
            time.sleep(rate_limit_delay)
    
    return progress, pages_downloaded, False


print("Incremental extraction function defined")

Incremental extraction function defined


In [9]:
def run_extraction_pipeline(
    tickers: list,
    progress: pd.DataFrame,
    api_key: str,
    start_date: str,
    end_date: str,
    max_pages: int = 10,
    rate_limit_delay: int = 15
) -> pd.DataFrame:
    """
    Run the extraction pipeline for all tickers.
    Stops if rate limit is hit.
    
    Returns:
        Updated progress DataFrame
    """
    print("\n" + "=" * 60)
    print("STARTING EXTRACTION PIPELINE")
    print("=" * 60)
    print(f"Date range: {start_date} to {end_date}")
    print(f"Tickers: {len(tickers)}")
    print(f"Max pages per ticker: {max_pages}")
    print("=" * 60)
    
    total_pages = 0
    tickers_completed = 0
    
    for i, ticker in enumerate(tickers):
        print(f"\n[{i+1}/{len(tickers)}] Processing {ticker}")
        
        progress, pages, hit_rate_limit = extract_ticker_sentiment_incremental(
            ticker=ticker,
            progress=progress,
            api_key=api_key,
            start_date=start_date,
            end_date=end_date,
            max_pages=max_pages,
            rate_limit_delay=rate_limit_delay
        )
        
        total_pages += pages
        
        if pages > 0:
            tickers_completed += 1
        
        if hit_rate_limit:
            print("\n" + "!" * 60)
            print("RATE LIMIT REACHED - STOPPING PIPELINE")
            print("Progress has been saved. Re-run notebook to continue.")
            print("!" * 60)
            break
        
        # Delay between tickers
        if pages > 0 and i < len(tickers) - 1:
            print(f"  Waiting {rate_limit_delay}s before next ticker...")
            time.sleep(rate_limit_delay)
    
    print("\n" + "=" * 60)
    print("EXTRACTION SUMMARY")
    print("=" * 60)
    print(f"  Total pages downloaded: {total_pages}")
    print(f"  Tickers with new data: {tickers_completed}")
    print(f"  Progress saved to: {PROGRESS_FILE}")
    
    return progress


print("Extraction pipeline defined")

Extraction pipeline defined


In [10]:
# Run the extraction pipeline
progress_df = run_extraction_pipeline(
    tickers=TICKERS,
    progress=progress_df,
    api_key=API_KEY,
    start_date=START_DATE,
    end_date=END_DATE,
    max_pages=MAX_PAGES_PER_TICKER,
    rate_limit_delay=RATE_LIMIT_DELAY
)


STARTING EXTRACTION PIPELINE
Date range: 2023-01-01 to 2026-01-01
Tickers: 16
Max pages per ticker: 10

[1/16] Processing NSEI
  [NSEI] Starting from page 3
    Fetching page 3...
    API info: We have detected your API key as X1F6BEW1FN2W3PWF and our standard API rate limi...
    Error on page 3, skipping ticker

[2/16] Processing RELIANCE
  [RELIANCE] Starting from page 3
    Fetching page 3...
    API info: Thank you for using Alpha Vantage! Please consider spreading out your free API r...
    Error on page 3, skipping ticker

[3/16] Processing TCS
  [TCS] Starting from page 3
    Fetching page 3...
    API info: Thank you for using Alpha Vantage! Please consider spreading out your free API r...
    Error on page 3, skipping ticker

[4/16] Processing INFY
  [INFY] Starting from page 3
    Fetching page 3...
    API info: Thank you for using Alpha Vantage! Please consider spreading out your free API r...
    Error on page 3, skipping ticker

[5/16] Processing HDFCBANK
  [HDFCBANK] S

In [11]:
# Show current progress
print("\nCurrent Progress Status:")
print(progress_df.to_string())


Current Progress Status:
        Ticker  Last_Page_Downloaded        Last_Run_Date
0         NSEI                     2  2026-04-03 14:32:58
1     RELIANCE                     2  2026-04-03 14:33:14
2          TCS                     2  2026-04-03 14:33:29
3         INFY                     2  2026-04-03 14:33:45
4     HDFCBANK                     2  2026-04-03 14:34:01
5    ICICIBANK                     2  2026-04-03 14:34:16
6         SBIN                     2  2026-04-03 14:34:32
7           LT                     2  2026-04-03 14:34:47
8          ITC                     2  2026-04-03 14:35:03
9    KOTAKBANK                     1  2026-04-03 14:27:25
10    AXISBANK                     1  2026-04-03 14:27:40
11  BAJFINANCE                     1  2026-04-03 14:27:56
12  HINDUNILVR                     1  2026-04-03 14:28:11
13  ASIANPAINT                     1  2026-04-03 14:28:27
14      MARUTI                     1  2026-04-03 14:28:42
15   SUNPHARMA                     1  2026-04-

---
## 7. Parse Cached Data

In [12]:
def parse_time_published(time_str: str) -> datetime:
    """
    Parse Alpha Vantage time_published format to datetime.
    Format: YYYYMMDDTHHMMSS
    """
    try:
        return datetime.strptime(time_str[:15], "%Y%m%dT%H%M%S")
    except (ValueError, TypeError):
        return None


def parse_cached_responses(tickers: list) -> pd.DataFrame:
    """
    Parse all cached JSON responses into a DataFrame.
    
    Returns:
        DataFrame with article-level data
    """
    all_articles = []
    
    for ticker in tickers:
        page = 1
        while is_page_cached(ticker, page):
            try:
                data = load_from_cache(ticker, page)
                feed = data.get("feed", [])
                
                for item in feed:
                    time_published = item.get("time_published")
                    sentiment_score = item.get("overall_sentiment_score")
                    
                    if time_published and sentiment_score is not None:
                        parsed_time = parse_time_published(time_published)
                        if parsed_time:
                            all_articles.append({
                                "Ticker": ticker,
                                "DateTime": parsed_time,
                                "Date": parsed_time.date(),
                                "Sentiment_Score": float(sentiment_score),
                                "Title": item.get("title", "")[:100]
                            })
            except Exception as e:
                print(f"Error parsing {ticker} page {page}: {e}")
            
            page += 1
    
    if all_articles:
        return pd.DataFrame(all_articles)
    else:
        return pd.DataFrame()


# Parse all cached data
print("Parsing cached responses...")
raw_articles_df = parse_cached_responses(TICKERS)
print(f"Total articles parsed: {len(raw_articles_df)}")

Parsing cached responses...
Total articles parsed: 100


In [13]:
# Preview raw article data
if not raw_articles_df.empty:
    print("Raw Article Data Preview:")
    print(raw_articles_df.head(10).to_string())
else:
    print("No articles found in cache.")

Raw Article Data Preview:
  Ticker            DateTime        Date  Sentiment_Score                                                                                                 Title
0   INFY 2026-01-01 07:09:06  2026-01-01         0.103371                     Infosys Shares Rise 1% on First Day of New Year 2026 After Five-Day Losing Streak
1   INFY 2026-01-01 04:09:46  2026-01-01        -0.427677                                                          Infosys Ltd eases for fifth straight session
2   INFY 2026-01-01 03:00:00  2026-01-01         0.867116            Infosys Ltd Sees Robust Trading Activity Amid Positive Momentum and Institutional Interest
3   INFY 2026-01-01 02:09:46  2026-01-01         0.022879  TCS, HCLTech, Infosys, Coforge To Announce Quarterly Earnings On These Dates — IT Q3 Results Calenda
4   INFY 2025-12-31 06:54:24  2025-12-31         0.251772                       The Truth About Infosys Ltd: Why Wall Street Can’t Ignore This Quiet Tech Beast
5   INFY 2025-

---
## 8. Daily Aggregation

In [14]:
def aggregate_daily_sentiment(articles_df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate article-level sentiment to daily level per ticker.
    """
    if articles_df.empty:
        return pd.DataFrame(columns=["Date", "Ticker", "News_Sentiment", "News_Count"])
    
    df = articles_df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    
    daily_sentiment = df.groupby(["Date", "Ticker"]).agg(
        News_Sentiment=("Sentiment_Score", "mean"),
        News_Count=("Sentiment_Score", "count")
    ).reset_index()
    
    daily_sentiment = daily_sentiment.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    daily_sentiment["News_Sentiment"] = daily_sentiment["News_Sentiment"].round(6)
    daily_sentiment["News_Count"] = daily_sentiment["News_Count"].astype(int)
    
    return daily_sentiment


# Aggregate to daily level
daily_sentiment_df = aggregate_daily_sentiment(raw_articles_df)
print(f"Daily aggregated records: {len(daily_sentiment_df)}")

Daily aggregated records: 17


In [15]:
# Preview daily sentiment
if not daily_sentiment_df.empty:
    print("Daily Sentiment Preview:")
    print(daily_sentiment_df.head(20).to_string())

Daily Sentiment Preview:
         Date Ticker  News_Sentiment  News_Count
0  2025-12-15   INFY        0.489572           2
1  2025-12-16   INFY       -0.042942          10
2  2025-12-17   INFY       -0.170764           6
3  2025-12-18   INFY        0.094947           8
4  2025-12-19   INFY        0.108024          12
5  2025-12-20   INFY        0.215130           4
6  2025-12-21   INFY        0.273414           2
7  2025-12-22   INFY        0.201181          10
8  2025-12-23   INFY        0.121258           6
9  2025-12-24   INFY        0.403082           2
10 2025-12-26   INFY        0.323627           2
11 2025-12-27   INFY       -0.282460           2
12 2025-12-28   INFY        0.001697           8
13 2025-12-29   INFY        0.280295           6
14 2025-12-30   INFY       -0.006307           8
15 2025-12-31   INFY        0.257521           4
16 2026-01-01   INFY        0.145344           8


---
## 9. Save Output

In [16]:
# Save aggregated data
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

if not daily_sentiment_df.empty:
    # Remove duplicates
    daily_sentiment_df = daily_sentiment_df.drop_duplicates(
        subset=["Date", "Ticker"], keep="first"
    ).reset_index(drop=True)
    
    # Final column order
    final_df = daily_sentiment_df[["Date", "Ticker", "News_Sentiment", "News_Count"]]
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"Dataset saved to: {OUTPUT_FILE}")
    print(f"Records saved: {len(final_df)}")
else:
    print("No data to save.")

Dataset saved to: ..\Market_Data\raw\news_sentiment.csv
Records saved: 17


---
## 10. Validation

In [17]:
# Final validation
print("=" * 60)
print("FINAL VALIDATION REPORT")
print("=" * 60)

if OUTPUT_FILE.exists():
    validation_df = pd.read_csv(OUTPUT_FILE, parse_dates=["Date"])
    
    print(f"\n1. Dataset Shape: {validation_df.shape}")
    print(f"\n2. Unique Tickers: {validation_df['Ticker'].nunique()}")
    print(f"   {sorted(validation_df['Ticker'].unique().tolist())}")
    print(f"\n3. Date Range:")
    print(f"   Start: {validation_df['Date'].min()}")
    print(f"   End: {validation_df['Date'].max()}")
    print(f"\n4. Missing Values:")
    print(validation_df.isnull().sum().to_string())
    print(f"\n5. Sample Rows:")
    print(validation_df.head(10).to_string())

# Cache summary
cache_files = list(CACHE_DIR.glob("*.json"))
print(f"\n6. Cache Files: {len(cache_files)}")

# Progress summary
print(f"\n7. Progress File: {PROGRESS_FILE}")
print(progress_df.to_string())

print("\n" + "=" * 60)

FINAL VALIDATION REPORT

1. Dataset Shape: (17, 4)

2. Unique Tickers: 1
   ['INFY']

3. Date Range:
   Start: 2025-12-15 00:00:00
   End: 2026-01-01 00:00:00

4. Missing Values:
Date              0
Ticker            0
News_Sentiment    0
News_Count        0

5. Sample Rows:
        Date Ticker  News_Sentiment  News_Count
0 2025-12-15   INFY        0.489572           2
1 2025-12-16   INFY       -0.042942          10
2 2025-12-17   INFY       -0.170764           6
3 2025-12-18   INFY        0.094947           8
4 2025-12-19   INFY        0.108024          12
5 2025-12-20   INFY        0.215130           4
6 2025-12-21   INFY        0.273414           2
7 2025-12-22   INFY        0.201181          10
8 2025-12-23   INFY        0.121258           6
9 2025-12-24   INFY        0.403082           2

6. Cache Files: 25

7. Progress File: ..\Market_Data\raw\news_sentiment\api_progress.csv
        Ticker  Last_Page_Downloaded        Last_Run_Date
0         NSEI                     2  2026-04-03

---
## Summary

This notebook implements **incremental data collection** with progress tracking:

### Progress Tracking
- Progress saved to `Market_Data/raw/news_sentiment/api_progress.csv`
- Tracks: Ticker | Last_Page_Downloaded | Last_Run_Date
- Updated after each successful API call
- Notebook can be stopped and resumed without losing progress

### Caching
- Raw JSON responses saved to `Market_Data/raw/news_sentiment/raw_json/`
- Filename: `NEWS_SENTIMENT_<TICKER>_page<NNN>.json`

### Rate Limit Handling
- Stops pipeline when rate limit is hit
- Saves progress before stopping
- Re-run notebook to continue from where it left off

### Output
- Aggregated CSV: `Market_Data/raw/news_sentiment.csv`
- Columns: Date | Ticker | News_Sentiment | News_Count

**Note:** News data should be lagged by 1 day when merging with stock data (handled in notebook 08)